In [1]:
import sys
from pathlib import Path

# 1. Define the absolute path to the directory containing your Python file
# Example: If your script is one level up in a folder named 'src'
module_path = str(Path.cwd().parent) 

# 2. Add that path to the system path if it isn't already there
if module_path not in sys.path:
    sys.path.append(module_path)



In [17]:
from trend_down_stocks import * 

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import config


In [4]:
    ticker = 'INTU'
    ticker = ticker.upper()
    plot_file = STOOQ_SAVE_DIR / f"debug_trend_down_{ticker}.png"

    print("=" * 72)
    print(f"Debug trend_down_stocks — {ticker}")
    print("=" * 72)
    print(f"  trend_down_thresh           = {config.trend_down_thresh}")
    print(f"  trend_down_history          = {config.trend_down_history}d")
    print(f"  trending_down_suppression   = {config.trending_down_suppression}d")
    print("=" * 72)

    daily_df = load_daily_prices()
    ticker_df = daily_df[daily_df["ticker"] == ticker].copy()
   

Debug trend_down_stocks — INTU
  trend_down_thresh           = 0.1
  trend_down_history          = 15d
  trending_down_suppression   = 30d
Loading daily prices from /Users/juanliu/Workspace/git_test/bargain_stocks/data/stock_Stooq_daily_US ...
Loaded 19,757,756 daily rows for 9,038 tickers (1962-01-02 → 2026-07-22)


In [5]:
    trending_down_stocks = find_trending_down_stocks(ticker_df)
    trending_down_stocks = suppress_similar_tuples(trending_down_stocks)

    events = pd.DataFrame(
        trending_down_stocks,
        columns=["ticker", "date", "price", "historical_average"],
    )
    print(f"\n{ticker} trend-down events after suppression: {len(events)}")
    if not events.empty:
        print(events.to_string(index=False))



Flagged 254 trend-down events (thresh=0.1, history=15d)
Suppressed 199 near-duplicate events (window=30d); kept 55

INTU trend-down events after suppression: 55
ticker       date     price  historical_average
  INTU 1993-09-24   2.38553            2.707405
  INTU 1994-03-28   2.85397            3.247542
  INTU 1994-05-09   2.19670            2.501040
  INTU 1995-04-28   5.07768            5.860004
  INTU 1995-10-04   6.59907            7.361127
  INTU 1995-11-21  10.31940           11.940064
  INTU 1996-01-04  10.20970           11.402067
  INTU 1996-02-14   8.40864            9.355144
  INTU 1996-06-26   6.80613            7.689283
  INTU 1996-09-04   4.87948            5.625817
  INTU 1996-10-17   3.89835            4.510240
  INTU 1996-12-12   4.50102            5.176790
  INTU 1997-01-27   4.57336            5.145685
  INTU 1997-02-28   3.39405            3.934290
  INTU 1998-01-09   5.27587            5.862604
  INTU 1998-05-15   6.87837            7.768607
  INTU 1998-07-27   8.4

In [8]:
events_INTU = pd.read_csv('/Users/juanliu/Workspace/git_test/bargain_stocks/data/stock_Stooq_daily_US/derived_data/debug_trending_down_stocks.csv')
events_INTU[-5:]



,ticker,date,price,historical_average,future_date,future_price,price_change
50,INTU,2024-05-30,562.97,644.547000,2024-09-03,624.81,1.109846
51,INTU,2026-01-14,566.60,643.432000,2026-04-17,393.25,0.694052
52,INTU,2026-02-13,399.40,445.815455,2026-05-18,403.16,1.009414
53,INTU,2026-04-09,361.69,420.272000,2026-07-13,289.76,0.801128
54,INTU,2026-05-21,307.07,391.133636,NaN,NaN,NaN


In [9]:
events_ALL = pd.read_csv('/Users/juanliu/Workspace/git_test/bargain_stocks/data/stock_Stooq_daily_US/derived_data/trending_down_stocks.csv')
len(events_ALL)

177893

In [11]:
events_ALL[events_ALL['ticker']=='INTU']['price_change'].mean()

np.float64(1.084448032621465)

In [27]:
events_ALL_since2010= events_ALL[(events_ALL['date']>='2010-01-01') & (events_ALL['future_date']!= np.nan)]
events_ALL_since2010['price_change'].mean() , events_ALL_since2010['price_change'].median(), (events_ALL_since2010['price_change']>1).mean()

(np.float64(1.0514909684201381),
 0.9857785580487264,
 np.float64(0.45695053119735846))

In [28]:
(events_ALL_since2010['price_change']>1).value_counts()

price_change
False    80916
True     68087
Name: count, dtype: int64

In [26]:
        with_future = events_ALL.dropna(subset=["price_change"]).copy()
        with_future["date"] = pd.to_datetime(with_future["date"])
        with_future = with_future[with_future["date"] >= "2010-01-01"]
 
        print(
            f"Price-change stats since 2010 (future/current): "
            f"median={with_future['price_change'].median():.5f}, "
            f"mean={with_future['price_change'].mean():.5f}, "
            f"pct_rebound(>1)="
            f"{(with_future['price_change'] > 1).mean() * 100:.1f}%"
        )

Price-change stats since 2010 (future/current): median=0.98578, mean=1.05149, pct_rebound(>1)=47.9%


In [29]:
(with_future['price_change'] > 1).value_counts()

price_change
False    73995
True     68087
Name: count, dtype: int64

In [31]:
-len(with_future)+len(events_ALL_since2010)

6921

In [32]:
events_INTU[-5:]

,ticker,date,price,historical_average,future_date,future_price,price_change
50,INTU,2024-05-30,562.97,644.547000,2024-09-03,624.81,1.109846
51,INTU,2026-01-14,566.60,643.432000,2026-04-17,393.25,0.694052
52,INTU,2026-02-13,399.40,445.815455,2026-05-18,403.16,1.009414
53,INTU,2026-04-09,361.69,420.272000,2026-07-13,289.76,0.801128
54,INTU,2026-05-21,307.07,391.133636,NaN,NaN,NaN


In [33]:
daily_df = load_daily_prices()
ticker_df = daily_df[daily_df["ticker"] == ticker].copy()
 

Loading daily prices from /Users/juanliu/Workspace/git_test/bargain_stocks/data/stock_Stooq_daily_US ...
Loaded 19,757,756 daily rows for 9,038 tickers (1962-01-02 → 2026-07-22)


In [35]:
ticker_df[ticker_df['date']=='2026-01-14']

,ticker,date,close_price
9299696,INTU,2026-01-14,566.6


In [46]:
ticker_df[(ticker_df['date']>='2025-12-29') & (ticker_df['date']<='2026-01-13')]['close_price'].mean()

np.float64(646.2245454545455)